# Ghaderi, Crochet & Petersen (2026) from dandiset 001530: population decoding and clustering

**Paper.** Ghaderi P, Crochet S, Petersen CCH (2026), *Nature Communications* 17, 5982,
[doi:10.1038/s41467-026-73622-y](https://doi.org/10.1038/s41467-026-73622-y); claims written from the
version of record (PMC13346651), main text, legends and Methods.

**Dandiset.** [DANDI:001530](https://dandiarchive.org/dandiset/001530/0.260605.0601), version
0.260605.0601, the only published version (the draft has the same assets and was last modified
with the publication); the paper cites no version. The data check, the trial and unit selection
and the claims on behaviour, optogenetics and single neurons are in `reproduce_ghaderi_2026.ipynb`;
this notebook loads the same sessions through the same helper.

**Outside inputs.** `data_helpers/Area_list.mat` (CCF labels of each area) and
`processed_data/Data_Clustering.mat` (the paper's cluster labels, linked by claim
`cluster-label-link` in the first notebook), from Zenodo doi:10.5281/zenodo.17973874.

**Run size.** The paper's decoding repeats each fit over 100 neuron subsets (Fig. 5b) or 5 runs
(Fig. 7d), and its GMM uses 1,000 initialisations per k. A full run of the code's decoder takes
about 25 ms per bin and subset here (timed below), and the Methods' grid search about ten times
more, so both are run with fewer repetitions (constants in the next cell); each section prints
what was run and the estimate for the full run.

| Claim | Here |
|---|---|
| fig5b-context-decoding-delay | tested, fewer repetitions |
| fig5b-decoding-time-course | tested, fewer repetitions |
| fig5c-cluster-removal, fig7d-context-decoding-after-whisker, fig4-29-clusters | `decoding_clusters_whisker.ipynb` |
| fig5d-null-space-decoding | blocked (see the record) |

In [ ]:
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from scipy import stats

import ghaderi as g

warnings.filterwarnings("ignore")
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 400)

N_JOBS = 4
REPS_CODE = 5           # neuron subsets per session, code's decoder (paper: 100)
REPS_TC = 3             # the same for 5 to 45 neurons (time course)
REPS_METHODS = 1        # neuron subsets per session, Methods' grid search (paper: 100)
BASE_STEP_METHODS = 5   # Methods' reading: every 5th baseline bin of -1 to 0 s
RUNS_7D = 1             # Fig. 7d runs (paper: 5)
GMM_INITS = 5           # GMM initialisations per k (paper: 1,000)

started = time.time()
S = g.load_sessions(workers=4)
print(f"dandiset {g.DANDISET_ID} version {g.DANDISET_VERSION}: loaded {len(S)} sessions in {time.time() - started:.0f} s")

In [ ]:
rows = []
for a in g.AREAS:
    for s, cols in g.area_sessions(S, a, 5):
        q = g.quiet_mask(s)
        rows.append({"area": a, "session": s["session"], "subject": s["subject"], "neurons": len(cols),
                     "quiet Hit": int((g.trial_mask(s, 1, True) & q).sum()),
                     "quiet Nogo CR": int((g.trial_mask(s, 3, False) & q).sum())})
dsess = pd.DataFrame(rows)
print(dsess.to_string(index=False))
print("\nArea-sessions with at least 5 neurons:", dsess.groupby("area").size().loc[g.AREAS].to_dict())
print("with at least 50 neurons:", dsess[dsess.neurons >= 50].groupby("area").size().loc[g.AREAS].to_dict(),
      "(paper, Fig. 5b: A1 8, wS1 17, wS2 19, wM2 18, ALM 16)")
print("mice with at least 50 neurons:", dsess[dsess.neurons >= 50].groupby("area").subject.nunique().loc[g.AREAS].to_dict(),
      "(paper: 6, 9, 9, 8, 6)")
print("Spike counts per 10 ms bin, range:", min(int(s["c10"].min()) for s in S if s["c10"].size), "to", max(int(s["c10"].max()) for s in S if s["c10"].size))

The sessions with at least 50 neurons per area match the paper's n for Fig. 5b (8, 17, 19, 18, 16
sessions); quiet Hit and Nogo CR trials per session are listed above, some as few as 7.

## fig5b-context-decoding-delay

> With 50 neurons per area, context (Hit against Nogo-tone Whisker CR) is decoded in the last 200 ms of
> the delay above baseline in every area, but far better in wM2 (64 +/- 1%) and ALM (61 +/- 1%) than in A1
> (52 +/- 1%), wS1 (52 +/- 1%) and wS2 (52 +/- 0.4%).

As the criterion: per 10 ms bin, a linear SVM on the counts of 50 random neurons, majority class
undersampled, z-scored, stratified 5-fold, mean over subsets. Delay = bins labelled 0.8 to 1.0 s,
baseline -1 to 0 s (Methods) and -0.8 to 0 s (code). Reading (b) is the code's nu-SVC, reading (a)
the Methods' C-SVM with nested grid search; (a) is run on one subset per session and on every 5th
baseline bin.

In [ ]:
DELAY = g.bin_index(0.8, 1.0)
BASE = g.bin_index(-1.0, 0.0)
BASE_CODE = g.bin_index(-0.8, 0.0)
CUE_A, CUE_B = g.bin_index(0.01, 0.03), g.bin_index(0.01, 0.2)


def session_classes(s, cols, rule="median", counts="c10"):
    q = g.quiet_mask(s, rule)
    X = s[counts][:, :, cols]
    return X[g.trial_mask(s, 1, True) & q], X[g.trial_mask(s, 3, False) & q]


tasks = []
for a in g.AREAS:
    for k, (s, cols) in enumerate(g.area_sessions(S, a, 50)):
        X1, X2 = session_classes(s, cols)
        tasks.append((a, s["session"], X1, X2, k))

t0 = time.time()
X1, X2 = tasks[0][2], tasks[0][3]
g.decode_session(X1, X2, DELAY[:4], 50, 1, "code", 0)
per_bin_code = (time.time() - t0) / 4
t0 = time.time()
g.decode_session(X1, X2, DELAY[:2], 50, 1, "methods", 0)
per_bin_methods = (time.time() - t0) / 2
bins_main = np.concatenate([BASE, CUE_B, DELAY])
print(f"Warm-up timing on one session (includes start-up): code {per_bin_code * 1000:.0f} ms, Methods {per_bin_methods * 1000:.0f} ms per bin and subset.")

t0 = time.time()
acc_code = Parallel(n_jobs=N_JOBS)(delayed(g.decode_session)(t[2], t[3], bins_main, 50, REPS_CODE, "code", 1000 + i)
                                   for i, t in enumerate(tasks))
t_code = time.time() - t0
print(f"code's decoder, {REPS_CODE} subsets: {t_code:.0f} s; 100 subsets would take about {t_code * 100 / REPS_CODE / 60:.0f} min")
bins_methods = np.concatenate([BASE[::BASE_STEP_METHODS], DELAY])
t0 = time.time()
acc_meth = Parallel(n_jobs=N_JOBS)(delayed(g.decode_session)(t[2], t[3], bins_methods, 50, REPS_METHODS, "methods", 2000 + i)
                                   for i, t in enumerate(tasks))
t_meth = time.time() - t0
print(f"Methods' grid search, {REPS_METHODS} subset(s) and {len(bins_methods)} bins: {t_meth:.0f} s; 100 subsets on all "
      f"{len(bins_main)} bins would take about {t_meth * 100 / REPS_METHODS * len(bins_main) / len(bins_methods) / 3600:.1f} h")
ix = {b: i for i, b in enumerate(bins_main)}
ixm = {b: i for i, b in enumerate(bins_methods)}
dec = pd.DataFrame([{"area": t[0], "session": t[1],
                     "delay_b": acc[[ix[b] for b in DELAY]].mean(), "base_b": acc[[ix[b] for b in BASE]].mean(),
                     "base_code_b": acc[[ix[b] for b in BASE_CODE]].mean(),
                     "cue_a_b": acc[[ix[b] for b in CUE_A]].mean(), "cue_b_b": acc[[ix[b] for b in CUE_B]].mean(),
                     "delay_a": am[[ixm[b] for b in DELAY]].mean(), "base_a": am[[ixm[b] for b in BASE[::BASE_STEP_METHODS]]].mean()}
                    for t, acc, am in zip(tasks, acc_code, acc_meth)])

In [ ]:
exp5b = {"A1": (52, 1), "wS1": (52, 1), "wS2": (52, 0.4), "wM2": (64, 1), "ALM": (61, 1)}
rows = []
for reading, dcol, bcols in (("b, code", "delay_b", ("base_b", "base_code_b")), ("a, Methods", "delay_a", ("base_a",))):
    for a in g.AREAS:
        sub = dec[dec.area == a]
        row = {"reading": reading, "area": a, "sessions": len(sub), "delay %": sub[dcol].mean(),
               "SEM": sub[dcol].std(ddof=1) / np.sqrt(len(sub)), "paper": f"{exp5b[a][0]} +/- {exp5b[a][1]}",
               "within": abs(sub[dcol].mean() - exp5b[a][0]) <= exp5b[a][1]}
        for bc in bcols:
            row[f"{bc} %"] = sub[bc].mean()
            row[f"p vs {bc}"] = stats.wilcoxon(sub[dcol], sub[bc]).pvalue
        rows.append(row)
res5b = pd.DataFrame(rows)
print(res5b.round(4).to_string(index=False))
for reading in ("b, code", "a, Methods"):
    sub = res5b[res5b.reading == reading].set_index("area")
    pcol = "p vs base_b" if reading.startswith("b") else "p vs base_a"
    order = min(sub.loc[["wM2", "ALM"], "delay %"]) > max(sub.loc[["A1", "wS1", "wS2"], "delay %"])
    print(f"{reading}: within tolerance {int(sub.within.sum())} of 5; all five tests p < 0.05: {bool((sub[pcol] < 0.05).all())}; "
          f"wM2 and ALM above A1, wS1, wS2: {order}")
print("\nPer mouse (reading b, delay accuracy):")
dec["subject"] = dec.session.str[4:9]
print(dec.pivot_table(index="subject", columns="area", values="delay_b").round(1).to_string())

**Result: partially reproduced.** Under both readings of the SVM the delay accuracy is above
baseline in all five areas (Wilcoxon p 0.0078 or less with the code's decoder, 0.027 or less with
the grid search) and wM2 and ALM exceed A1, wS1 and wS2, but
the values are partly out of tolerance. With the code's nu-SVC (5 subsets per session): A1 52.6, wS1
51.9, wS2 51.4, wM2 62.0 and ALM 58.9% (2 of 5 within; the paper 52, 52, 52, 64 and 61). With the
Methods' grid search (one subset): 53.1, 52.3, 52.0, 63.1 and 58.5% (3 of 5 within). Both readings
give the same outcome. Fewer subsets than the paper's 100 were run (run times above).

## fig5b-decoding-time-course

> Context decoding is at chance in the baseline, highest in A1 right after the cue, and during the
> delay falls in A1, wS1 and wS2 but rises in wM2 and ALM; with fewer neurons accuracy drops but wM2 and
> ALM stay above A1, wS1 and wS2.

Baseline, cue and delay accuracies for 50 neurons come from the run above; for 5 to 45 neurons the
delay bins are decoded with the code's decoder (REPS_TC subsets, set above) and, for 5 and 25 neurons, the
Methods' grid search (one subset).

In [ ]:
counts_n = list(range(5, 50, 5))
t0 = time.time()
jobs = [(n, i, t) for n in counts_n for i, t in enumerate(tasks)]
out = Parallel(n_jobs=N_JOBS)(delayed(g.decode_session)(t[2], t[3], DELAY, n, REPS_TC, "code", 3000 + 100 * n + i)
                              for n, i, t in jobs)
tc_rows = [{"n": n, "area": t[0], "session": t[1], "reading": "b", "delay": acc.mean()} for (n, i, t), acc in zip(jobs, out)]
jobs_m = [(n, i, t) for n in (5, 25) for i, t in enumerate(tasks)]
out_m = Parallel(n_jobs=N_JOBS)(delayed(g.decode_session)(t[2], t[3], DELAY, n, REPS_METHODS, "methods", 4000 + 100 * n + i)
                                for n, i, t in jobs_m)
tc_rows += [{"n": n, "area": t[0], "session": t[1], "reading": "a", "delay": acc.mean()} for (n, i, t), acc in zip(jobs_m, out_m)]
tc_rows += [{"n": 50, "area": r.area, "session": r.session, "reading": "b", "delay": r.delay_b} for r in dec.itertuples()]
tc_rows += [{"n": 50, "area": r.area, "session": r.session, "reading": "a", "delay": r.delay_a} for r in dec.itertuples()]
tcd = pd.DataFrame(tc_rows)
print(f"time-course runs: {time.time() - t0:.0f} s")
tab = tcd.pivot_table(index=["reading", "n"], columns="area", values="delay")[g.AREAS]
print(tab.round(2).to_string())
ok4 = {(r, n): bool(min(row["wM2"], row["ALM"]) > max(row["A1"], row["wS1"], row["wS2"])) for (r, n), row in tab.iterrows()}
print("\n(4) wM2 and ALM above A1, wS1 and wS2 at each count:", ok4)
m50 = dec.groupby("area")[["base_b", "cue_a_b", "cue_b_b", "delay_b"]].mean().loc[g.AREAS]
print("\n50 neurons, code's decoder, mean over sessions (%):")
print(m50.round(2).to_string())
print("(1) baseline within 45-55 in every area:", bool(((m50.base_b - 50).abs() <= 5).all()))
for cue in ("cue_a_b", "cue_b_b"):
    print(f"(2) {cue}: highest in A1: {m50[cue].idxmax() == 'A1'}; "
          f"(3) delay below cue in A1, wS1, wS2: {[bool(m50.loc[a, 'delay_b'] < m50.loc[a, cue]) for a in ('A1', 'wS1', 'wS2')]}, "
          f"above in wM2, ALM: {[bool(m50.loc[a, 'delay_b'] > m50.loc[a, cue]) for a in ('wM2', 'ALM')]}")

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3))
for a in g.AREAS:
    v = tab.loc["b"][a]
    ax.plot(v.index, v.values, "-o", ms=3, label=a)
ax.axhline(50, color="k", lw=0.5)
ax.set_xlabel("neurons")
ax.set_ylabel("delay accuracy (%)")
ax.legend(fontsize=7)
plt.show()
print(f"Plotted delay accuracies span {tab.loc['b'].values.min():.1f} to {tab.loc['b'].values.max():.1f}%.")

**Result: reproduced.** With the code's decoder and 50 neurons, (1) baseline accuracy is 49.98 to
50.09% in every area; (2) the early cue accuracy is highest in A1 under both cue windows (70.3% over 0
to 30 ms, 65.6% over 0 to 200 ms; at most 57.4% and 57.9% elsewhere); (3) the delay accuracy is below
the cue accuracy in A1, wS1 and wS2 and above it in wM2 and ALM, under both windows; (4) wM2 and ALM
exceed A1, wS1 and wS2 at every neuron count from 5 to 50 (at 5 neurons by 0.47 to 0.51 points), and
also with the Methods' grid search at 5, 25 and 50 neurons. Parts (1) to (3) were run with the code's
decoder only.

In [ ]:
print(f"notebook ran in {(time.time() - started) / 60:.1f} min")

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning
and the differences from the paper's method.

| Claim | Outcome |
|---|---|
| fig5b-context-decoding-delay | partially reproduced |
| fig5b-decoding-time-course | reproduced |

**How the analysis differs from the paper's.** 5 random subsets of 50 neurons per session (3 for 5
to 45 neurons) in place of 100 with the code's decoder, and one subset on every 5th baseline bin with
the Methods' grid search, whose grid (C from 10^-3 to 10^3) is the claims file's. Quiet trials by the
Methods' median absolute deviation. sklearn's NuSVC and SVC (both libsvm) stand in for the MATLAB
libsvm interface; folds and undersampling use their own random draws.

**What was found about the dataset.** The area-sessions with at least 50 neurons are exactly the
paper's (8, 17, 19, 18 and 16 sessions in 6, 9, 9, 8 and 6 mice).